# 3-Class Forgery Classifier (Real / Edited / Deepfake)

Thin notebook -- logic lives in `data/*.py` and `model/*.py`. Sections: Data, Model, Train, Eval.
See `architecture_decisions.md` for the design and `data_download.md` / `model_code.md` for the implementation instructions this follows.

## Setup

In [ ]:
!pip install -q facenet-pytorch timm kaggle datasets

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
# Confirm GPU tier per model_code.md section 0 -- A100 gets the full 3-branch
# config below as-is; T4/L4 need the fallback config from architecture_decisions.md's GPU-tier table.

In [ ]:
# Mount Drive if running in Colab and you want raw/checkpoint persistence
# from google.colab import drive
# drive.mount('/content/drive')

import os
# os.environ['KAGGLE_API_TOKEN'] = ''  # set before running the Data section
# os.environ['HF_TOKEN'] = ''          # optional, only needed if you hit HF rate limits

## Data
Downloads the three raw sources, then runs uniform MTCNN face detect+crop, split, and manifest generation.

In [ ]:
from data.download import download_coco_ai, download_casia

download_coco_ai(n_pairs=3000)
download_casia()

In [ ]:
import subprocess, sys
# face_filter.py is a script (not just importable functions) -- run it directly so its
# per-source logging (seen/detected/rejected) prints in full, including the deepfake
# survival-rate guard.
subprocess.run([sys.executable, 'data/face_filter.py'], check=True)

In [ ]:
# Class balance check -- must feed the class-weighted loss below, never hardcoded proportions
import pandas as pd
from config import MANIFEST_PATH

manifest = pd.read_csv(MANIFEST_PATH)
print(manifest.groupby(['class', 'split']).size())

## Model
Instantiate the 3-branch fusion model and dry-run a single batch through the full pipeline before committing to the full training loop.

In [ ]:
from config import DEVICE, EMBED_DIM, CLASSES
from model.dataset import get_dataloader
from model.fusion import ForgeryClassifier

model = ForgeryClassifier(embed_dim=EMBED_DIM, num_classes=len(CLASSES)).to(DEVICE)

dry_loader = get_dataloader('train', batch_size=8, shuffle=True, num_workers=2)
batch = next(iter(dry_loader))
rgb = batch['rgb'].to(DEVICE)
fft_mag = batch['fft_mag'].to(DEVICE)
srm = batch['srm_residual'].to(DEVICE)

logits, gate_weights = model(rgb, fft_mag, srm)
print('logits:', logits.shape, 'gate_weights:', gate_weights.shape)
assert logits.shape == (8, len(CLASSES))
assert gate_weights.shape == (8, 3)
print('Dry run OK -- shapes check out.')

## Train

In [ ]:
from model.train import train

best_path, best_macro_f1 = train(
    epochs=18,
    batch_size=64,  # drop to 32 on OOM before touching resolution, per model_code.md section 4
    lr=3e-4,
    weight_decay=1e-4,
    num_workers=4,
    patience=5,
)
print(best_path, best_macro_f1)

## Eval
Macro-F1, per-class precision/recall/ROC-AUC, 3x3 confusion matrix, and the paired Grad-CAM + gate-weight explainability dump.

In [ ]:
from model.eval import load_model, compute_metrics, print_report, explainability_dump
from model.dataset import get_dataloader, ForgeryDataset

eval_model = load_model(str(best_path))
val_loader = get_dataloader('val', batch_size=64, shuffle=False, num_workers=4)
metrics = compute_metrics(eval_model, val_loader)
print_report(metrics)

In [ ]:
val_dataset = ForgeryDataset('val')
explainability_results = explainability_dump(eval_model, val_dataset, samples_per_class=3)

In [ ]:
# Visualize a few Grad-CAM heatmaps alongside their gate weights
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, min(4, len(explainability_results)), figsize=(16, 4))
for ax, r in zip(axes, explainability_results[:4]):
    img = Image.open(r['path']).convert('RGB')
    ax.imshow(img)
    ax.imshow(r['gradcam'], cmap='jet', alpha=0.4)
    gate_str = ', '.join(f"{k}:{v:.2f}" for k, v in r['gate_weights'].items())
    ax.set_title(f"{r['true_class']}\n{gate_str}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()